In [1]:
import sys, os

sys.path.insert(0, os.path.abspath("/home/ElasticNotebook"))
%load_ext ElasticNotebook
from elastic.core.common.pandas import compare_df
import pickle
import cudf

Enabled rmm statistics


In [2]:
%LoadCheckpoint /home/colinc/code/dias-benchmarks/notebooks/joshuaswords/netflix-data-visualization/src/rewritten_cpu/o4_mini_high/checkpoints/post_cell_23_try_0.pickle

trying: ['i_2', 'i_1']
me:  10
me:  10
trying: ['i_2', 'i_1']
me:  10
me:  10
trying: ['labelPadding', 'upperLimit']
me:  19
me:  19
trying: ['labelPadding', 'upperLimit']
me:  19
me:  19
trying: ['width']
me:  19
trying: ['mf_ratio']
me:  9
trying: ['df_heatmap']
me:  23
trying: ['df_movies']
me:  24
trying: ['x']
me:  9
trying: ['loli']
me:  25
trying: ['dt']
me:  1
trying: ['ratings_ages']
me:  11
trying: ['warnings']
me:  1
trying: ['matplotlib']
me:  20
trying: ['time']
me:  0
trying: ['MultiLabelBinarizer']
me:  20
trying: ['cmap']
me:  20
trying: ['np']
me:  0
trying: ['df']


me:  21
trying: ['data_q2q3_ratio']
me:  13
trying: ['data_q2q3']
me:  13
trying: ['r']
me:  9
trying: ['null_rate']
me:  4
trying: ['data']
me:  25
trying: ['y']
me:  9
trying: ['i']
me:  4
trying: ['country_order']
me:  13
trying: ['ordered_df_rev']
me:  25
trying: ['indexes']
me:  19
trying: ['stats']
me:  25
trying: ['ordered_df']
me:  25
trying: ['max']
me:  19
trying: ['data_sub2']
me:  19
trying: ['genre_heatmap']
me:  21
trying: ['order']
me:  14
trying: ['pd']
me:  0
trying: ['rating_order']
me:  14
trying: ['df_polar']
me:  19
trying: ['lowerLimit']
me:  19
trying: ['movie']
me:  15
trying: ['color_map']
me:  19
trying: ['mf']
me:  15
trying: ['heights']
me:  19
trying: ['tv']
me:  15
trying: ['data_sub']
me:  19
trying: ['month_order']
me:  17
trying: ['angles']
me:  19
trying: ['df_tv']
me:  25
trying: ['df_loli']
me:  25
trying: ['slope']
me:  19
trying: ['factor']
me:  3
Declaring variable time
Declaring variable np
Declaring variable pd
Declaring variable dt
Declaring va

In [3]:
%%RecordEvent
%%time
### cell 24 ###

# Optimize filtering and grouping for speed and memory
use_modin = os.environ.get("IREWR_WITH_MODIN") == "True"

# 1) Fast boolean filter using .isin
mask_countries = df['first_country'].isin(['USA', 'India'])
us_ind = df.loc[mask_countries]

# 2) If running under Modin, convert to pandas once
if use_modin:
    df = df._to_pandas()

# 3) Re-compute mask on the (possibly converted) DataFrame
mask_group = df['first_country'].isin(['USA', 'India'])

# 4) Group on the reduced frame; use .size + unstack(fill_value=0) instead of value_counts+fillna
#    cast to float to match original fillna behavior
data_sub = (
    df.loc[mask_group, ['first_country', 'year_added']]
      .groupby(['first_country', 'year_added'])
      .size()
      .unstack(fill_value=0)
      .astype(float)
      .loc[['USA', 'India']]
      .cumsum(axis=0)
      .T
)

# 5) If Modin, wrap outputs as pandas.DataFrame to match original
if use_modin:
    df = pd.DataFrame(df)
    data_sub = pd.DataFrame(data_sub)

CPU times: user 18 ms, sys: 150 µs, total: 18.1 ms
Wall time: 18.1 ms


In [4]:
%Checkpoint /home/colinc/code/dias-benchmarks/notebooks/joshuaswords/netflix-data-visualization/src/rewritten_cpu/o4_mini_high/checkpoints/post_cell_24_try_0.pickle

migration speed (bps): 716338998.673481


---------------------------
variables to migrate:
r 158
mf_ratio 93
i_2 53
i_1 53
month_order 152
df_movies 39516434
pd 72
mask_countries 395566
genre_heatmap 144
data_q2q3 967
null_rate 32
country_order 703
width 24
i 60
data_q2q3_ratio 879
labelPadding 28
df_polar 1170
lowerLimit 28
data_sub 352
color_map 152
use_modin 24
heights 208
ordered_df 801
us_ind 31880514
data 641
factor 28
mask_group 395566
angles 152
df_tv 17311634
upperLimit 28
np 72
loli 801
indexes 152
df_loli 14145924
max 32
ordered_df_rev 801
order 1108
slope 32
stats 5234
rating_order 184
data_sub2 1202
tv 980
movie 980
time 72
mf 366
dt 72
ratings_ages 640
warnings 72
df 56827294
x 158
matplotlib 72
MultiLabelBinarizer 1072
cmap 48
df_heatmap 589
y 28
---------------------------
variables to recompute:
[]
---------------------------
cells to recompute:
[]
Checkpoint saved to: /home/colinc/code/dias-benchmarks/notebooks/joshuaswords/netflix-data-visualization/src/rewritten_cpu/o4_mini_high/checkpoints/post_cell_24_tr

In [5]:
%PrintCellInfo opt_cell_exec_info

======= Cell 0 =======
Input variables []
Active variables ['df']
Intermediate variables []
Future variables []
Modified dataframes
======= Cell 1 =======
Input variables ['df']
Active variables ['df']
Intermediate variables ['factor']
Future variables []
Modified dataframes
======= Cell 2 =======
Input variables ['df']
Active variables []
Intermediate variables ['null_rate', 'i']
Future variables ['df']
Modified dataframes
======= Cell 3 =======
Input variables ['df']
Active variables ['df']
Intermediate variables []
Future variables []
Modified dataframes
  df
    Input columns: set()
    Changed columns: {'description', 'country', 'director', 'listed_in', 'title', 'release_year', 'rating', 'duration', 'cast', 'date_added', 'type', 'show_id'}
    Created columns: set()
    Deleted columns: set()
======= Cell 4 =======
Input variables ['df']
Active variables []
Intermediate variables []
Future variables ['df']
Modified dataframes
======= Cell 5 =======
Input variables ['df']
Active va

In [6]:
with open(
    "/home/colinc/code/dias-benchmarks/notebooks/joshuaswords/netflix-data-visualization/src/opt_cell_exec_info_24_try_0.pkl",
    "wb",
) as f:
    pickle.dump(opt_cell_exec_info[24], f)

In [7]:
opt_output = Out.get(4)

In [8]:
us_ind_opt = us_ind
data_sub_opt = data_sub
%LoadCheckpoint /home/colinc/code/dias-benchmarks/notebooks/joshuaswords/netflix-data-visualization/src/annotated_cpu/checkpoints/post_cell_24.pickle

import numpy as np
import cudf
from elastic.core.common.pandas import is_type_styler

is_orig_output_pd = isinstance(orig_output, (pd.Series, pd.DataFrame, pd.Index))
is_opt_output_pd = isinstance(opt_output, (pd.Series, pd.DataFrame, pd.Index))
is_orig_output_array = isinstance(
    orig_output, (cudf.pandas._wrappers.numpy.ndarray, np.ndarray)
)
is_opt_output_array = isinstance(
    opt_output, (cudf.pandas._wrappers.numpy.ndarray, np.ndarray)
)
is_orig_output_styler = is_type_styler(type(orig_output))
is_opt_output_styler = is_type_styler(type(opt_output))
if is_orig_output_styler and is_opt_output_styler:
    assert orig_output.to_html() == opt_output.to_html()
elif is_orig_output_styler:
    assert orig_output.to_html() == opt_output.to_html()
elif is_opt_output_styler:
    assert opt_output.to_html() == orig_output

if is_orig_output_pd and is_opt_output_pd:
    assert orig_output.equals(opt_output)
# TODO(jie): this is a hack.
elif (
    (is_orig_output_pd or is_opt_output_pd)
    and (is_orig_output_array or is_opt_output_array)
) or (is_orig_output_array and is_opt_output_array):
    assert list(orig_output) == list(opt_output)
else:
    assert orig_output == opt_output

trying: ['labelPadding', 'upperLimit']
me:  36
me:  36
trying: ['labelPadding', 'upperLimit']
me:  36
me:  36
trying: ['i_1', 'i_2']
me:  18
me:  18
trying: ['i_1', 'i_2']
me:  18
me:  18
trying: ['df_polar']
me:  36
trying: ['month_order']
me:  32
trying: ['max']
me:  36
trying: ['ratings_ages']
me:  20
trying: ['slope']
me:  36
trying: ['color_map']
me:  36
trying: ['pd']
me:  0
trying: ['genre_heatmap']
me:  40
trying: ['country_order']
me:  24
trying: ['us_ind']


me:  50
trying: ['data_q2q3']
me:  24
trying: ['data_q2q3_ratio']
me:  24
trying: ['mf_ratio']
me:  16
trying: ['data_sub']
me:  50
trying: ['time']
me:  0
trying: ['df_movies']


me:  46
trying: ['warnings']
me:  1
trying: ['dt']
me:  1
trying: ['y']
me:  16
trying: ['x']
me:  16
trying: ['orig_output']
me:  3
trying: ['ordered_df']
me:  48
trying: ['ordered_df_rev']
me:  48
trying: ['loli']
me:  48
trying: ['rating_order']
me:  26
trying: ['r']
me:  16
trying: ['df_tv']


me:  48
trying: ['df_loli']


me:  48
trying: ['df']


me:  40
trying: ['tv']
me:  28
trying: ['data']
me:  48
trying: ['mf']
me:  28
trying: ['movie']
me:  28
trying: ['np']
me:  0
trying: ['df_heatmap']
me:  44
trying: ['width']
me:  36
trying: ['indexes']
me:  36
trying: ['order']
me:  26
trying: ['null_rate']
me:  6
trying: ['angles']
me:  36
trying: ['i']
me:  6
trying: ['heights']
me:  36
trying: ['matplotlib']
me:  38
trying: ['data_sub2']
me:  36
trying: ['MultiLabelBinarizer']
me:  38
trying: ['lowerLimit']
me:  36
trying: ['cmap']
me:  38
trying: ['factor']
me:  4


Declaring variable pd
Declaring variable time
Declaring variable np
Declaring variable warnings
Declaring variable dt
Declaring variable orig_output
Declaring variable factor
Declaring variable null_rate
Declaring variable i
Declaring variable mf_ratio
Declaring variable y
Declaring variable x
Declaring variable r
Declaring variable i_1
Declaring variable i_2
Declaring variable i_1
Declaring variable i_2
Declaring variable ratings_ages
Declaring variable country_order
Declaring variable data_q2q3
Declaring variable data_q2q3_ratio
Declaring variable rating_order
Declaring variable order
Declaring variable tv
Declaring variable mf
Declaring variable movie
Declaring variable month_order
Declaring variable labelPadding
Declaring variable upperLimit
Declaring variable labelPadding
Declaring variable upperLimit
Declaring variable df_polar
Declaring variable max
Declaring variable slope
Declaring variable color_map
Declaring variable width
Declaring variable indexes
Declaring variable angles